# AB03 Ilseung — knee RA / RD check

Focused check of `ab03_ilseung_knee_0p8mps_ra_exo_on` and `..._rd_exo_on` using the same **no-LPF** methodology as `compare_processed_knee_exo_id`.

Processed data: `/media/metamobility3/Samsung_T52/Results/processed/AB03_Ilseung` (original).

- **Sync**: GPIO falling-edge alignment
- **Inputs**: raw, **no input LPF** (Vicon: B-spline velocity; encoder: logged `model_in_knee_vel_raw`)
- **Output**: raw TCN → zero-phase 6 Hz LPF + global `MODEL_OUT_ALIGN_LAG_SAMPLES`
- **GT**: Vicon ID/mass − `cmd_R`/mass, zero-phase 6 Hz LPF, per-trial encoder–IK xcorr ID shift
- **AB03 corrections**: none in the batch notebook (`GT_OFFSET` / `ENCODER_OFFSET` empty for this subject); toggles below are left for experimentation

Checkpoint: `runs/0707_knee_finetune_balanced_lg_ra_rd/best_model.pt`


In [ ]:
import io
import inspect
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_SUBJECT_DIR = Path('/media/metamobility3/Samsung_T52/Results/processed/AB03_Ilseung')
TELEMETRY_ROOT = PROJECT_ROOT

TRIAL_STEMS = [
    'ab03_ilseung_knee_0p8mps_ra_exo_on',
    'ab03_ilseung_knee_0p8mps_rd_exo_on',
]
SUBJECT_TOKEN = 'ab03_ilseung'
SUBJECT_NAME = 'AB03_Ilseung'
SUBJECT_MASS_KG = 84.4

EXO_KIND = 'knee-exo'
JOINT = 'knee_angle_r'
JOINT_LABEL = 'Knee R'
MOMENT_COL = 'knee_angle_r_moment'
MOCAP_FS_HZ = 1000.0
LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
GT_LPF_MODE = 'zero_phase'
MODEL_LPF_MODE = 'zero_phase'
MODEL_OUT_ALIGN_LAG_SAMPLES = -6
ENCODER_IK_XCORR_MAX_LAG = 300

CHECKPOINT_PATH = PROJECT_ROOT / 'runs' / '0707_knee_finetune_balanced_lg_ra_rd' / 'best_model.pt'

TRIM_START_SEC = 10.0
TRIM_END_SEC = 10.0

# Optional per-trial corrections (empty for AB03 in batch notebook; edit to experiment).
GT_OFFSET_NMPKG_BY_STEM: Dict[str, float] = {}
ENCODER_OFFSET_DEG_BY_STEM: Dict[str, float] = {}

# Defaults: match no-LPF batch replay (xcorr ID shift on; GT/encoder offsets off).
APPLY_GT_OFFSET = False
APPLY_ENCODER_OFFSET = False

print(f'Subject: {SUBJECT_NAME} | joint: {JOINT_LABEL} | mass={SUBJECT_MASS_KG} kg')
print(f'Processed: {PROCESSED_SUBJECT_DIR}')
print(f'Trials: {TRIAL_STEMS}')
print(f'Checkpoint: {CHECKPOINT_PATH} ({"OK" if CHECKPOINT_PATH.is_file() else "MISSING"})')
print(f'APPLY_GT_OFFSET={APPLY_GT_OFFSET} | APPLY_ENCODER_OFFSET={APPLY_ENCODER_OFFSET}')
print(f'MODEL_OUT_ALIGN_LAG_SAMPLES={MODEL_OUT_ALIGN_LAG_SAMPLES:+d}')


In [ ]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    return rmse, float(1.0 - ss_res / (ss_tot + 1e-12))


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    return {
        'npz': TELEMETRY_ROOT / f'{trial_stem}.npz',
        'mocap': PROCESSED_SUBJECT_DIR / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv',
        'id': PROCESSED_SUBJECT_DIR / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto',
        'ik': PROCESSED_SUBJECT_DIR / EXO_KIND / 'ik' / f'{cond}_{speed}_ik.mot',
        'cond': cond,
        'speed': speed,
    }


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines='skip')
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    return np.arange(len(df)) / fs, df['jet'].to_numpy(dtype=float)


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    arr = np.asarray(gpio, dtype=np.float64)
    g_range = arr.max() - arr.min()
    return arr if g_range <= 0 else (arr - arr.min()) / g_range


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    above = np.asarray(signal, dtype=np.float64) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError('No GPIO key in npz')


def extract_applied_cmd_nm(npz) -> Tuple[np.ndarray, str]:
    for k in ('cmd_R', 'cmd_L'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No cmd_R/cmd_L; keys={sorted(npz.files)}')


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def _fill_nan_1d(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def _shift_samples_1d(x: np.ndarray, lag_samples: int) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64)
    out = np.full_like(arr, np.nan)
    lag = int(lag_samples)
    if lag > 0:
        if lag < len(arr):
            out[:-lag] = arr[lag:]
    elif lag < 0:
        lag = -lag
        if lag < len(arr):
            out[lag:] = arr[:-lag]
    else:
        out = arr.copy()
    return _fill_nan_1d(out)


def sync_to_wave_t(t_src, y_src, wave: Dict, *, t_src_on_npz_clock: bool = False) -> np.ndarray:
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    t_src = np.asarray(t_src, dtype=np.float64)
    if t_src_on_npz_clock:
        n = int(min(len(t_aligned), len(t_src), len(y_src)))
        t_aligned = t_aligned[:n]
        t_src = t_src[:n]
        y_src = y_src[:n]
        t_ref = t_src + float(wave['offset_s'])
    else:
        t_ref = t_src
    return _fill_nan_1d(np.interp(t_aligned, t_ref, y_src, left=np.nan, right=np.nan))


def analysis_trim_mask(t: np.ndarray, trim_start_s=TRIM_START_SEC, trim_end_s=TRIM_END_SEC) -> np.ndarray:
    t = np.asarray(t, dtype=np.float64)
    t_rel = t - np.nanmin(t)
    return (t_rel >= trim_start_s) & (t_rel <= (np.nanmax(t_rel) - trim_end_s))


def model_out_nmpkg_lpf(raw: np.ndarray, fs_hz: float) -> np.ndarray:
    return lpf_nan(raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, MODEL_LPF_MODE)


def load_gpio_sync_data(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap', 'id', 'ik'):
        if not paths[key].exists():
            raise FileNotFoundError(paths[key])
    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = np.asarray(d['time'], dtype=np.float64) if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]
    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)
    return {
        'trial': trial_stem,
        'paths': paths,
        'npz': d,
        'gpio_key': gpio_key,
        'offset_s': offset_s,
        'idx_exo': idx_exo,
        'idx_mocap': idx_mocap,
        't_npz': t_raw,
        'gpio_npz': gpio,
        't_mocap': t_mocap,
        'gpio_mocap_norm': normalize_gpio(gpio_mocap),
        't_npz_aligned': t_raw + float(offset_s) if offset_s is not None else t_raw.copy(),
    }


def _load_vicon_knee_ik(stem: str) -> Tuple[np.ndarray, np.ndarray]:
    paths = resolve_trial_paths(stem)
    cols, data = read_sto(paths['ik'])
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_rad = np.deg2rad(data[:, cols.index('knee_angle_r')].astype(np.float64))
    return t_mocap, knee_rad


def _vicon_ik_velocity_spline(t_mocap: np.ndarray, knee_rad: np.ndarray) -> np.ndarray:
    from scipy.interpolate import splrep, splev
    t_mocap = np.asarray(t_mocap, dtype=np.float64)
    knee_rad = np.asarray(knee_rad, dtype=np.float64)
    if len(t_mocap) < 5:
        dt = 1.0 / infer_fs_hz(t_mocap)
        vel = np.zeros_like(knee_rad)
        if len(knee_rad) > 1:
            vel[1:] = (knee_rad[1:] - knee_rad[:-1]) / dt
        return vel
    tck = splrep(t_mocap, knee_rad, s=0, k=3)
    return np.asarray(splev(t_mocap, tck, der=1), dtype=np.float64)


def _encoder_vicon_best_lag_samples(
    enc_rad: np.ndarray,
    vicon_rad: np.ndarray,
    *,
    max_lag: int = ENCODER_IK_XCORR_MAX_LAG,
    mask: Optional[np.ndarray] = None,
) -> int:
    enc = np.asarray(enc_rad, dtype=np.float64)
    vicon = np.asarray(vicon_rad, dtype=np.float64)
    best_lag, best_score = 0, -np.inf
    for lag in range(-int(max_lag), int(max_lag) + 1):
        shifted = _shift_samples_1d(enc, lag)
        s = shifted[mask] if mask is not None else shifted
        v = vicon[mask] if mask is not None else vicon
        mm = np.isfinite(s) & np.isfinite(v)
        if mm.sum() < 100:
            continue
        score = float(np.corrcoef(s[mm], v[mm])[0, 1])
        if score > best_score:
            best_score, best_lag = score, lag
    return int(best_lag)


def build_vicon_replay_inputs_no_lpf(t_mocap, knee_rad, wave: Dict):
    knee_vel = _vicon_ik_velocity_spline(t_mocap, knee_rad)
    angle_sync = sync_to_wave_t(t_mocap, knee_rad, wave, t_src_on_npz_clock=False)
    vel_sync = sync_to_wave_t(t_mocap, knee_vel, wave, t_src_on_npz_clock=False)
    return angle_sync, vel_sync


def build_encoder_replay_inputs_no_lpf(stem: str, wave: Dict):
    d = np.load(str(TELEMETRY_ROOT / f'{stem}.npz'), allow_pickle=True)
    t_npz = np.asarray(d['time'], dtype=np.float64)
    ang_raw = vel_raw = None
    ang_key = vel_key = ''
    for key in ('model_in_knee_angle_raw', 'knee_angle_r'):
        if key in d.files:
            ang_raw, ang_key = np.asarray(d[key], dtype=np.float64), key
            break
    for key in ('model_in_knee_vel_raw', 'model_in_knee_vel_raw_r'):
        if key in d.files:
            vel_raw, vel_key = np.asarray(d[key], dtype=np.float64), key
            break
    if vel_raw is None and 'gyro_shank_r' in d.files and 'gyro_thigh_r' in d.files:
        vel_raw = np.asarray(d['gyro_shank_r'], dtype=np.float64) - np.asarray(d['gyro_thigh_r'], dtype=np.float64)
        vel_key = 'gyro_shank_r-gyro_thigh_r'
    if ang_raw is None or vel_raw is None:
        raise KeyError(f'Missing encoder angle/velocity in {stem}; keys={sorted(d.files)}')
    angle_sync = sync_to_wave_t(t_npz, ang_raw, wave, t_src_on_npz_clock=True)
    vel_sync = sync_to_wave_t(t_npz, vel_raw, wave, t_src_on_npz_clock=True)
    if APPLY_ENCODER_OFFSET:
        offset_deg = float(ENCODER_OFFSET_DEG_BY_STEM.get(stem, 0.0))
        if offset_deg != 0.0:
            angle_sync = angle_sync + np.deg2rad(offset_deg)
    return angle_sync, vel_sync, ang_key, vel_key


def rebuild_gt_xcorr_shift(stem: str, wave: Dict, id_shift_samples: int, n: int) -> np.ndarray:
    paths = resolve_trial_paths(stem)
    d = np.load(str(paths['npz']), allow_pickle=True)
    applied_nm, _ = extract_applied_cmd_nm(d)
    t_aligned = np.asarray(wave['t'], dtype=np.float64)[:n]
    applied_nm = np.asarray(applied_nm, dtype=np.float64)[:n]
    fs_hz = infer_fs_hz(t_aligned)
    cols, id_data = read_sto(paths['id'])
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(MOMENT_COL)]
    t_id_query = t_aligned + float(id_shift_samples) / fs_hz
    id_nm_raw = np.interp(t_id_query, t_id, id_moment_nm, left=np.nan, right=np.nan)
    net_raw = id_nm_raw / SUBJECT_MASS_KG - applied_nm / SUBJECT_MASS_KG
    gt = lpf_nan(net_raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, GT_LPF_MODE)
    if APPLY_GT_OFFSET:
        gt = gt + float(GT_OFFSET_NMPKG_BY_STEM.get(stem, 0.0))
    return gt


sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN  # noqa: E402


def _tcn_ctor_kwargs(cfg: dict) -> dict:
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != 'self'}
    return {k: v for k, v in cfg.items() if k in allowed}


@torch.no_grad()
def run_knee_tcn_inference(model, angle, vel, window_size, device):
    angle = np.asarray(angle, dtype=np.float32)
    vel = np.asarray(vel, dtype=np.float32)
    n = int(min(len(angle), len(vel)))
    pred = np.zeros(n, dtype=np.float32)
    model.eval()
    for t in range(n):
        start = max(0, t - window_size + 1)
        valid = t - start + 1
        x = np.zeros((2, window_size), dtype=np.float32)
        x[0, -valid:] = angle[start : t + 1]
        x[1, -valid:] = vel[start : t + 1]
        xt = torch.from_numpy(x).unsqueeze(0).to(device=device, dtype=torch.float32)
        pred[t] = float(model(xt)[0, 0, -1].item())
    return pred


def load_replay_model(ckpt_path: Path, device: str):
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    model_cfg = ckpt['model_config']
    window_size = int(ckpt.get('window_size', 100))
    model = TCN(**_tcn_ctor_kwargs(model_cfg)).eval()
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    return model, window_size


def build_subject_wave(stem: str, model, window_size: int, device: str) -> Dict:
    sync = load_gpio_sync_data(stem)
    if sync['offset_s'] is None:
        raise RuntimeError(f'{stem}: GPIO sync failed')

    d = sync['npz']
    applied_nm, applied_key = extract_applied_cmd_nm(d)
    n0 = min(len(sync['t_npz']), len(applied_nm))
    wave = {
        'trial': stem,
        't': sync['t_npz_aligned'][:n0].astype(np.float64),
        'offset_s': float(sync['offset_s']),
        'fs_hz': infer_fs_hz(sync['t_npz'][:n0]),
        'applied_key': applied_key,
        'gpio_key': sync['gpio_key'],
        'cond': resolve_trial_paths(stem)['cond'],
        'speed': resolve_trial_paths(stem)['speed'],
    }

    t_mocap, knee_rad = _load_vicon_knee_ik(stem)
    vicon_angle, vicon_vel = build_vicon_replay_inputs_no_lpf(t_mocap, knee_rad, wave)
    enc_angle, enc_vel, enc_key, enc_vel_key = build_encoder_replay_inputs_no_lpf(stem, wave)

    n = int(min(len(wave['t']), len(vicon_angle), len(enc_angle), len(vicon_vel), len(enc_vel)))
    wave['t'] = wave['t'][:n]
    vicon_angle, vicon_vel = vicon_angle[:n], vicon_vel[:n]
    enc_angle, enc_vel = enc_angle[:n], enc_vel[:n]
    fs_hz = float(wave['fs_hz'])

    trim_m = analysis_trim_mask(wave['t'])
    id_shift = _encoder_vicon_best_lag_samples(enc_angle, vicon_angle, mask=trim_m)
    gt = rebuild_gt_xcorr_shift(stem, wave, id_shift, n)

    vicon_pred_raw = run_knee_tcn_inference(model, vicon_angle, vicon_vel, window_size, device).astype(np.float64)
    enc_pred_raw = run_knee_tcn_inference(model, enc_angle, enc_vel, window_size, device).astype(np.float64)

    out = {
        **wave,
        'gt_nmpkg': gt,
        'vicon_angle_rad': vicon_angle,
        'vicon_vel_rad_s': vicon_vel,
        'encoder_angle_rad': enc_angle,
        'encoder_vel_rad_s': enc_vel,
        'encoder_angle_key': enc_key,
        'encoder_vel_key': enc_vel_key,
        'encoder_ik_xcorr_lag_samples': int(id_shift),
        'gt_offset_nmpkg': float(GT_OFFSET_NMPKG_BY_STEM.get(stem, 0.0)) if APPLY_GT_OFFSET else 0.0,
        'encoder_offset_deg': float(ENCODER_OFFSET_DEG_BY_STEM.get(stem, 0.0)) if APPLY_ENCODER_OFFSET else 0.0,
        'vicon_ik_model_out_nmpkg_raw': _shift_samples_1d(vicon_pred_raw, MODEL_OUT_ALIGN_LAG_SAMPLES),
        'encoder_replay_model_out_nmpkg_raw': _shift_samples_1d(enc_pred_raw, MODEL_OUT_ALIGN_LAG_SAMPLES),
        'vicon_ik_model_out_nmpkg': _shift_samples_1d(model_out_nmpkg_lpf(vicon_pred_raw, fs_hz), MODEL_OUT_ALIGN_LAG_SAMPLES),
        'encoder_replay_model_out_nmpkg': _shift_samples_1d(model_out_nmpkg_lpf(enc_pred_raw, fs_hz), MODEL_OUT_ALIGN_LAG_SAMPLES),
    }
    return out

print('Helpers ready.')


## 1. Load GPIO sync + replay RA / RD

Runs both trials through the no-LPF Vicon IK oracle and encoder replay pipeline against original processed IK/ID.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, window_size = load_replay_model(CHECKPOINT_PATH, device)
print(f'Model ready | window={window_size} | device={device}')

TRIAL_DATA: Dict[str, Dict] = {}
for stem in TRIAL_STEMS:
    wave = build_subject_wave(stem, model, window_size, device)
    TRIAL_DATA[stem] = wave
    trim_m = analysis_trim_mask(wave['t'])
    gt = wave['gt_nmpkg'][trim_m]
    rmse_e, r2_e = rmse_r2(gt, wave['encoder_replay_model_out_nmpkg'][trim_m])
    rmse_v, r2_v = rmse_r2(gt, wave['vicon_ik_model_out_nmpkg'][trim_m])
    ang_rmse, _ = rmse_r2(
        np.rad2deg(wave['vicon_angle_rad'][trim_m]),
        np.rad2deg(wave['encoder_angle_rad'][trim_m]),
    )
    print(
        f"OK  {stem} | GPIO={wave['offset_s']:+.3f}s | "
        f"ID xcorr={wave['encoder_ik_xcorr_lag_samples']:+d} | "
        f"GT offset={wave['gt_offset_nmpkg']:+.2f} | enc off={wave['encoder_offset_deg']:+.1f}° | "
        f"enc RMSE={rmse_e:.4f} R²={r2_e:.4f} | "
        f"vicon RMSE={rmse_v:.4f} R²={r2_v:.4f} | "
        f"ang RMSE={ang_rmse:.2f}° | n={len(wave['t'])}"
    )

rows = []
for stem, wave in TRIAL_DATA.items():
    trim_m = analysis_trim_mask(wave['t'])
    gt = wave['gt_nmpkg'][trim_m]
    rmse_e, r2_e = rmse_r2(gt, wave['encoder_replay_model_out_nmpkg'][trim_m])
    rmse_v, r2_v = rmse_r2(gt, wave['vicon_ik_model_out_nmpkg'][trim_m])
    ang_rmse, ang_r2 = rmse_r2(
        np.rad2deg(wave['vicon_angle_rad'][trim_m]),
        np.rad2deg(wave['encoder_angle_rad'][trim_m]),
    )
    vel_rmse, vel_r2 = rmse_r2(wave['vicon_vel_rad_s'][trim_m], wave['encoder_vel_rad_s'][trim_m])
    rows.append({
        'trial': stem,
        'cond': wave['cond'],
        'gpio_offset_s': wave['offset_s'],
        'id_xcorr_lag': wave['encoder_ik_xcorr_lag_samples'],
        'gt_offset_nmpkg': wave['gt_offset_nmpkg'],
        'encoder_offset_deg': wave['encoder_offset_deg'],
        'rmse_encoder': rmse_e,
        'r2_encoder': r2_e,
        'rmse_vicon': rmse_v,
        'r2_vicon': r2_v,
        'ang_rmse_deg': ang_rmse,
        'ang_r2': ang_r2,
        'vel_rmse_rad_s': vel_rmse,
        'vel_r2': vel_r2,
    })
metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))


## 2. Interactive compare (moments + inputs)

Dropdown switches RA ↔ RD. Time slider respects the 10 s end trim used for metrics.


In [ ]:
COLORS = {
    'GT': '#1e88e5',
    'Vicon IK': '#43a047',
    'Encoder': '#fb8c00',
}

plot_out = widgets.Output()
trial_dd = widgets.Dropdown(options=list(TRIAL_DATA.keys()), description='Trial:')
time_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f',
    layout=widgets.Layout(width='760px'),
)


def draw_compare(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t']) & (t_rel >= t_window[0]) & (t_rel <= t_window[1])
    gt = wave['gt_nmpkg'][m]
    enc = wave['encoder_replay_model_out_nmpkg'][m]
    vicon = wave['vicon_ik_model_out_nmpkg'][m]
    rmse_e, r2_e = rmse_r2(gt, enc)
    rmse_v, r2_v = rmse_r2(gt, vicon)

    vicon_ang = np.rad2deg(wave['vicon_angle_rad'][m])
    enc_ang = np.rad2deg(wave['encoder_angle_rad'][m])
    rmse_ang, r2_ang = rmse_r2(vicon_ang, enc_ang)
    vicon_vel = wave['vicon_vel_rad_s'][m]
    enc_vel = wave['encoder_vel_rad_s'][m]
    rmse_vel, r2_vel = rmse_r2(vicon_vel, enc_vel)

    fig, axs = plt.subplots(4, 1, figsize=(14, 13), sharex=True)

    ax = axs[0]
    ax.plot(t_rel[m], gt, color=COLORS['GT'], lw=2.0, label='GT (ID/mass − cmd/mass, 6 Hz z-p)')
    ax.plot(
        t_rel[m], vicon, color=COLORS['Vicon IK'], lw=1.6,
        label=f'Vicon IK oracle (RMSE={rmse_v:.3f}, R²={r2_v:.3f})',
    )
    ax.plot(
        t_rel[m], enc, color=COLORS['Encoder'], lw=1.5, ls='-.',
        label=f'Encoder replay (RMSE={rmse_e:.3f}, R²={r2_e:.3f})',
    )
    ax.set_ylabel('N·m/kg')
    ax.set_title('Model output vs GT (no input LPF; output 6 Hz z-p LPF)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    ax = axs[1]
    ax.plot(t_rel[m], (vicon - gt), color=COLORS['Vicon IK'], lw=1.3, label='Vicon − GT')
    ax.plot(t_rel[m], (enc - gt), color=COLORS['Encoder'], lw=1.3, ls='-.', label='Encoder − GT')
    ax.axhline(0, color='black', ls=':', lw=1)
    ax.set_ylabel('Residual (N·m/kg)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    ax = axs[2]
    ax.plot(t_rel[m], vicon_ang, color=COLORS['Vicon IK'], lw=1.5, label='Vicon IK knee_angle_r')
    ax.plot(
        t_rel[m], enc_ang, color=COLORS['Encoder'], lw=1.5, ls='-.',
        label=f"Encoder ({wave['encoder_angle_key']})",
    )
    ax.set_ylabel('Knee angle (deg)')
    ax.set_title(f'Inputs — angle (RMSE={rmse_ang:.2f}°, R²={r2_ang:.3f})')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    ax = axs[3]
    ax.plot(t_rel[m], vicon_vel, color=COLORS['Vicon IK'], lw=1.5, label='Vicon B-spline vel')
    ax.plot(
        t_rel[m], enc_vel, color=COLORS['Encoder'], lw=1.5, ls='-.',
        label=f"Encoder ({wave['encoder_vel_key']})",
    )
    ax.set_ylabel('rad/s')
    ax.set_xlabel('Time since sync (s)')
    ax.set_title(f'Inputs — velocity (RMSE={rmse_vel:.3f} rad/s, R²={r2_vel:.3f})')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.25)

    fig.suptitle(
        f"AB03 {wave['cond']} | GPIO={wave['offset_s']:+.3f}s | "
        f"ID xcorr={wave['encoder_ik_xcorr_lag_samples']:+d} | "
        f"GT off={wave['gt_offset_nmpkg']:+.2f} | enc off={wave['encoder_offset_deg']:+.1f}° | "
        f"out align={MODEL_OUT_ALIGN_LAG_SAMPLES:+d}",
        y=1.01,
    )
    fig.tight_layout()
    with plot_out:
        plot_out.clear_output(wait=True)
        plt.show()


def _init_slider(stem: str):
    wave = TRIAL_DATA[stem]
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t'])
    t_use = t_rel[m] if m.any() else t_rel
    time_slider.min = float(t_use[0])
    time_slider.max = float(t_use[-1])
    time_slider.step = max((time_slider.max - time_slider.min) / 500, 1e-3)
    time_slider.value = (time_slider.min, time_slider.max)


def _redraw(*_):
    draw_compare(TRIAL_DATA[trial_dd.value], time_slider.value)


def _on_trial(change):
    _init_slider(change['new'])
    _redraw()


trial_dd.observe(_on_trial, names='value')
time_slider.observe(_redraw, names='value')
_init_slider(trial_dd.value)
display(widgets.VBox([trial_dd, time_slider, plot_out]))
_redraw()


## 3. Residual output ↔ GT alignment sweep

Per-trial lag search on LPF'd replay outputs (±20 samples). Confirms whether the global `MODEL_OUT_ALIGN_LAG_SAMPLES=-6` is still reasonable for AB03 alone.


In [ ]:
SHIFT_SEARCH_SAMPLES = 20
_lags = np.arange(-SHIFT_SEARCH_SAMPLES, SHIFT_SEARCH_SAMPLES + 1)
_SHIFT_SERIES = (
    ('Vicon IK oracle', 'vicon_ik_model_out_nmpkg', COLORS['Vicon IK']),
    ('Encoder replay', 'encoder_replay_model_out_nmpkg', COLORS['Encoder']),
)

_per_trial_rows = []
for stem, wave in TRIAL_DATA.items():
    m = analysis_trim_mask(wave['t'])
    gt = wave['gt_nmpkg']
    fs = float(wave['fs_hz'])
    for label, col, _color in _SHIFT_SERIES:
        pred = wave[col]
        res = [rmse_r2(gt[m], _shift_samples_1d(pred, int(L))[m]) for L in _lags]
        rmse_by_lag = np.array([r[0] for r in res], dtype=np.float64)
        r2_by_lag = np.array([r[1] for r in res], dtype=np.float64)
        i0 = int(np.where(_lags == 0)[0][0])
        ibest = int(np.nanargmin(rmse_by_lag))
        _per_trial_rows.append({
            'trial': stem,
            'cond': wave['cond'],
            'series': label,
            'best_lag': int(_lags[ibest]),
            'best_lag_ms': float(_lags[ibest] / fs * 1000.0),
            'r2_now': float(r2_by_lag[i0]),
            'r2_shift': float(r2_by_lag[ibest]),
            'd_r2': float(r2_by_lag[ibest] - r2_by_lag[i0]),
            'rmse_now': float(rmse_by_lag[i0]),
            'rmse_shift': float(rmse_by_lag[ibest]),
            'd_rmse': float(rmse_by_lag[ibest] - rmse_by_lag[i0]),
        })

shift_df = pd.DataFrame(_per_trial_rows)
display(shift_df.round(4))

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, stem in zip(axs, TRIAL_STEMS):
    wave = TRIAL_DATA[stem]
    m = analysis_trim_mask(wave['t'])
    gt = wave['gt_nmpkg']
    for label, col, color in _SHIFT_SERIES:
        r2_by_lag = np.array([
            rmse_r2(gt[m], _shift_samples_1d(wave[col], int(L))[m])[1] for L in _lags
        ])
        ax.plot(_lags, r2_by_lag, color=color, lw=1.8, label=label)
        ib = int(np.nanargmax(r2_by_lag))
        ax.plot(_lags[ib], r2_by_lag[ib], 'o', color=color, ms=7)
    ax.axvline(0, color='black', ls=':', lw=1.0)
    ax.set_title(f"AB03 {wave['cond']}")
    ax.set_xlabel('Extra shift on replay output (samples)')
    ax.grid(alpha=0.25)
axs[0].set_ylabel('R² vs GT')
axs[0].legend(loc='lower center', fontsize=8)
fig.suptitle('AB03 residual alignment sweep (relative to current MODEL_OUT_ALIGN)', y=1.02)
fig.tight_layout()
plt.show()
